Configuração do Escopo de Banco de Dados

In [0]:
%sql
-- Camada Silver: Processamento e Normalização cadastro
-- **Descrição:** Este script lê os dados semiestruturados da camada Bronze, realiza o parsing dos campos JSON e popula o schema Silver de forma relacional.

-- Definição do catálogo de trabalho
USE CATALOG portfolio_mercado_livre;

-- Criação automática do Schema Silver se não existir
CREATE SCHEMA IF NOT EXISTS silver;

Criação e Carga da Tabela

In [0]:
%sql
--  1. Tabela trusted: Clientes (trusted_clientes)

-- Criação da tabela Delta estruturada na Silver
CREATE TABLE IF NOT EXISTS silver.trusted_clientes (
    sk_cliente string,
    id_cliente STRING,
    nickname_cliente STRING,
    tipo_documento STRING,
    estado_entrega STRING,
    cidade_entrega STRING,
    dh_insercao_silver TIMESTAMP
) USING DELTA;

-- Carga incremental/Sobrescrita da trusted Clientes
-- Fazendo o parsing dinâmico da coluna string JSON vinda da Bronze
INSERT OVERWRITE silver.trusted_clientes
SELECT DISTINCT
    hash(get_json_object(buyer, '$.id') ) as sk_cliente,
    get_json_object(buyer, '$.id') AS id_cliente,
    get_json_object(buyer, '$.nickname') AS nickname_cliente,
    get_json_object(buyer, '$.billing_info.doc_type') AS tipo_documento,
    get_json_object(buyer, '$.shipping_address.state') AS estado_entrega,
    get_json_object(buyer, '$.shipping_address.city') AS cidade_entrega,
    current_timestamp() AS dh_insercao_silver
FROM bronze.mercado_livre_vendas
WHERE get_json_object(buyer, '$.id') IS NOT NULL;

-- Prévia dos dados normalizados de clientes e regiões
SELECT * FROM silver.trusted_clientes LIMIT 5;

Criação e Carga da Tabela

In [0]:
%sql
-- 2. Tabela trusted: Produtos (trusted_produtos)

CREATE TABLE IF NOT EXISTS silver.trusted_produtos (
    sk_produto STRING,
    id_produto STRING,
    dh_insercao_silver TIMESTAMP
) USING DELTA;

-- Agrupando todos os produtos únicos da plataforma
INSERT OVERWRITE silver.trusted_produtos
SELECT DISTINCT
    hash(id_item_referencia) as sk_produto,
    id_item_referencia AS id_produto,
    current_timestamp() AS dh_insercao_silver
FROM bronze.mercado_livre_produtos
WHERE id_item_referencia IS NOT NULL;

SELECT * FROM silver.trusted_produtos LIMIT 5;

Criação e Carga da Tabela eventos_vendas (Com Explode de Itens)

In [0]:
%sql
-- MAGIC %md
-- MAGIC ### 3. Tabela evento: Vendas (evento_venda)

CREATE TABLE IF NOT EXISTS silver.evento_vendas (
    id_venda STRING,
    sk_cliente STRING,
    sk_produto STRING,
    quantidade_itens INT,
    valor_unitario DOUBLE,
    valor_total_venda DOUBLE,
    status_venda STRING,
    data_venda TIMESTAMP,
    dh_insercao_silver TIMESTAMP
) USING DELTA;

-- Carga da evento com extração do Array de Itens Comprados
WITH vendas_parsed AS (
    SELECT 
        id AS id_venda,
        hash(get_json_object(buyer, '$.id')) sk_cliente,
        status AS status_venda,
        CAST(total_amount AS DOUBLE) AS valor_total_venda,
        CAST(date_created AS TIMESTAMP) AS data_venda,
        -- Mapeia a estrutura do array JSON interno de itens da venda para que o SQL entenda
        from_json(order_items, 'ARRAY<STRUCT<item_id: STRING, quantity: INT, unit_price: DOUBLE>>') AS itens_array
    FROM bronze.mercado_livre_vendas
)
INSERT OVERWRITE silver.evento_vendas
SELECT 
    id_venda,
    sk_cliente,
    hash(item_desaninhado.item_id) AS sk_produto,
    item_desaninhado.quantity AS quantidade_itens,
    item_desaninhado.unit_price AS valor_unitario,
    valor_total_venda,
    status_venda,
    data_venda,
    current_timestamp() AS dh_insercao_silver
-- O EXPLODE quebra o Array de produtos, gerando linhas individuais se o cliente comprou mais de 1 produto diferente
FROM vendas_parsed
LATERAL VIEW explode(itens_array) AS item_desaninhado;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM silver.evento_vendas LIMIT 5;